# Model Serialization and Environment Capture

This notebook section is intended to be run **after all five classifiers have been trained** in the original ML development notebook.

It saves the fitted pipelines as `.joblib` files, reloads them to verify prediction consistency, and generates a version-pinned `requirements.txt` using the actual versions installed in this training environment.

**Important:** Run this in the same Python environment used to train the models. Do not install or upgrade packages between training and serialization.

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import joblib
import numpy as np
import pandas as pd
import sklearn

# Capture the actual versions from this model-training environment.
package_names = ["streamlit", "scikit-learn", "pandas", "numpy", "joblib", "matplotlib"]
versions = {}

for package in package_names:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = None

print("Python / ML environment versions")
print("scikit-learn:", sklearn.__version__)
for package, version in versions.items():
    print(f"{package}: {version if version else 'NOT INSTALLED'}")

## 1. Confirm the trained model variables

The following variables must already exist from the model-training section:
`logistic_pipeline`, `decision_tree_pipeline`, `knn_pipeline`, `naive_bayes_pipeline`, and `svm_pipeline`.

The pipelines are saved as-is, including their preprocessing steps such as `StandardScaler` where applicable.

In [ ]:
required_model_variables = [
    "logistic_pipeline",
    "decision_tree_pipeline",
    "knn_pipeline",
    "naive_bayes_pipeline",
    "svm_pipeline",
]

missing_variables = [name for name in required_model_variables if name not in globals()]
if missing_variables:
    raise NameError(
        "Train the five models first. Missing variables: " + ", ".join(missing_variables)
    )

models = {
    "logistic_regression": logistic_pipeline,
    "decision_tree": decision_tree_pipeline,
    "knn": knn_pipeline,
    "gaussian_nb": naive_bayes_pipeline,
    "svm": svm_pipeline,
}

print("All five trained model pipelines are available.")
for model_name, model in models.items():
    print(f"{model_name}: {type(model).__name__}")

## 2. Save the five trained models

The output directory follows the assignment repository structure: `model/`.

In [ ]:
model_dir = Path("model")
model_dir.mkdir(parents=True, exist_ok=True)

for model_name, model in models.items():
    output_path = model_dir / f"{model_name}.joblib"
    joblib.dump(model, output_path, compress=3)
    print(f"Saved: {output_path}")

print("\nSaved model files:")
for path in sorted(model_dir.glob("*.joblib")):
    print(f"{path} — {path.stat().st_size:,} bytes")

## 3. Reload and verify model predictions

This verification compares predictions from each in-memory fitted pipeline with predictions from its reloaded `.joblib` file. It requires the existing `X_test` variable from the train/test split.

In [ ]:
if "X_test" not in globals():
    raise NameError("X_test is not available. Run the train/test split cells first.")

verification_results = {}

for model_name, original_model in models.items():
    model_path = model_dir / f"{model_name}.joblib"
    loaded_model = joblib.load(model_path)

    original_predictions = original_model.predict(X_test)
    reloaded_predictions = loaded_model.predict(X_test)

    matches = np.array_equal(original_predictions, reloaded_predictions)
    verification_results[model_name] = matches
    print(f"{model_name}: {'PASS' if matches else 'FAIL'}")

if not all(verification_results.values()):
    raise AssertionError("One or more reloaded models produced different predictions.")

print("\nAll five saved models passed prediction consistency verification.")

## 4. Generate requirements.txt from the actual environment

This records the versions present in the same environment used to serialize the models. If Streamlit or Matplotlib is not installed, install it in the deployment environment separately and record its version after installation. Do not guess the scikit-learn version.

In [ ]:
required_for_app = ["streamlit", "scikit-learn", "pandas", "numpy", "joblib", "matplotlib"]
missing_packages = [name for name in required_for_app if not versions.get(name)]

if missing_packages:
    print("Not installed in this environment:", ", ".join(missing_packages))
    print("Install missing app dependencies, then rerun the version-capture and requirements cell.")
else:
    requirements_lines = [f"{name}=={versions[name]}" for name in required_for_app]
    requirements_path = Path("requirements.txt")
    requirements_path.write_text("\n".join(requirements_lines) + "\n", encoding="utf-8")
    print("Created requirements.txt with environment-matched versions:\n")
    print(requirements_path.read_text(encoding="utf-8"))

## 5. Final serialization checklist

- [ ] Five `.joblib` files exist in `model/`.
- [ ] Reloaded predictions match the original pipelines for all five models.
- [ ] `requirements.txt` contains the actual environment versions.
- [ ] Preserve this notebook as part of the project source code.

**Compatibility note:** Python package versions, especially scikit-learn and NumPy, should be kept consistent between model serialization and deployment. If deployment uses a different environment, test loading every model before publishing the app.